# Colab launcher

Thin launcher: clone the repo, install, run `train.py` with one config file. All logic lives in the repo.

Use a GPU runtime (Runtime > Change runtime type > T4 GPU). Checkpoints and results go to Google Drive, so after a disconnect, run all cells again with the same config and training resumes from `last.ckpt`.

In [ ]:
import os
import torch
from google.colab import drive

# Fail fast: on a CPU runtime training is far slower and the session is wasted.
assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU, then run again."

drive.mount("/content/drive")

REPO_URL = "https://github.com/ori112/representation-learning-course"
BRANCH = "midterm"
REPO_DIR = "/content/representation-learning-course"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}/midterm
!pip install -q -r requirements.txt
!nvidia-smi -L

In [ ]:
# Keep outputs on Drive so a disconnect does not lose checkpoints or results.
# The Tiny-ImageNet cache (dataset/) stays local: rebuilding it takes a few minutes.
DRIVE_DIR = "/content/drive/MyDrive/rl_midterm"
for folder in ("outputs", "results"):
    os.makedirs(f"{DRIVE_DIR}/{folder}", exist_ok=True)
    if not os.path.islink(folder):
        os.symlink(f"{DRIVE_DIR}/{folder}", folder)

In [ ]:
# Pick the experiment. One config file = one run.
CONFIG = "configs/speed_test.yaml"
!python train.py --config {CONFIG}

In [ ]:
# Run many configs in a row (a search or an ablation grid). Finished runs are skipped,
# so after a disconnect, run this cell again and it continues where it stopped.
import glob
CONFIG_GLOB = "configs/partA/robustness/*.yaml"
for config_path in sorted(glob.glob(CONFIG_GLOB)):
    run_name = os.path.basename(config_path)[:-len(".yaml")]
    if os.path.exists(f"results/{run_name}.json"):
        continue
    !python train.py --config {config_path}

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/tb_logs